# M-ViSER - Speech Emotion Recognition

**Repo**: https://github.com/Huu2412/M-ViSER

---
### Training Modes
| Stage | Mo ta | Lenh |
|---|---|---|
| **0** | End-to-End (Student + Teacher cung luc) | `--stage 0` |
| **1** | Chi train Teacher (Audio + Clean Text) | `--stage 1` |
| **2** | Student Distillation tu Teacher da freeze | `--stage 2 --teacher_ckpt ...` |

> **Recommended**: Chay Stage 1 truoc -> lay checkpoint -> Stage 2


In [ ]:
# ============================================================
# CELL 1: Clone repo (luon lay code moi nhat)
# ============================================================
import os

REPO_URL = 'https://github.com/Huu2412/M-ViSER.git'
REPO_DIR = '/kaggle/working/M-ViSER'

os.system(f'rm -rf {REPO_DIR}')
os.system(f'git clone {REPO_URL} {REPO_DIR}')

print('=== Latest 3 commits ===')
os.system(f'git -C {REPO_DIR} log --oneline -3')


In [ ]:
# ============================================================
# CELL 2: Cai dat dependencies
# ============================================================
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', f'{REPO_DIR}/requirements.txt'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])
else:
    print('Dependencies installed OK.')


In [ ]:
# ============================================================
# CELL 3: Kiem tra moi truong GPU
# ============================================================
import torch, os

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1e9
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {vram_gb:.1f} GB')
    print(f'CUDA version    : {torch.version.cuda}')

os.system('df -h /kaggle/working')


In [ ]:
# ============================================================
# CELL 4: Smoke Test (kiem tra forward + backward pass)
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

print(f'Working dir: {os.getcwd()}')
ret = os.system('python smoke_test.py')
print('\nSmoke test: PASSED' if ret == 0 else '\nSmoke test: FAILED')


In [ ]:
# ============================================================
# CELL 5A: Train -- End-to-End (Stage 0)
#   Student + Teacher cung hoc song song.
#   Su dung khi khong co thoi gian train 2 stage rieng.
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

os.system('python train.py --config config/config.yaml --stage 0')


In [ ]:
# ============================================================
# CELL 5B: Train -- Stage 1: Teacher Only
#   Chi train nhanh Teacher (Audio + Clean Text -> Emotion).
#   Checkpoint luu vao: checkpoints/stage1_teacher/best_model.pt
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

os.system('python train.py --config config/config.yaml --stage 1')


In [ ]:
# ============================================================
# CELL 5C: Train -- Stage 2: Student Distillation
#   Student (audio-only) hoc tu Teacher bi freeze.
#   Can chay Cell 5B truoc de co teacher checkpoint!
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

TEACHER_CKPT = f'{REPO_DIR}/checkpoints/stage1_teacher/best_model.pt'

if not os.path.exists(TEACHER_CKPT):
    print(f'Teacher checkpoint khong tim thay: {TEACHER_CKPT}')
    print('  --> Hay chay Cell 5B (Stage 1) truoc!')
else:
    print(f'Teacher checkpoint: {TEACHER_CKPT}')
    cmd = (
        f'python train.py '
        f'--config config/config.yaml '
        f'--stage 2 '
        f'--teacher_ckpt {TEACHER_CKPT}'
    )
    os.system(cmd)


In [ ]:
# ============================================================
# CELL 6: 5-Fold Cross-Validation
#   Chay toan bo 5 folds IEMOCAP speaker-independent.
#   Chinh bien FOLDS de chay 1 so fold cu the.
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

FOLDS = '1 2 3 4 5'  # doi thanh '1 2' de thu nhanh

os.system(f'python run_5fold.py --config config/config.yaml --folds {FOLDS}')


In [ ]:
# ============================================================
# CELL 7: Evaluate tren test set
# ============================================================
import sys, os
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Tu dong tim checkpoint tot nhat
BEST_CKPT = f'{REPO_DIR}/checkpoints/best_model.pt'
candidates = [
    f'{REPO_DIR}/checkpoints/stage2_student/best_model.pt',
    f'{REPO_DIR}/checkpoints/stage1_teacher/best_model.pt',
]
for c in candidates:
    if not os.path.exists(BEST_CKPT) and os.path.exists(c):
        BEST_CKPT = c

print(f'Evaluating checkpoint: {BEST_CKPT}')
os.system(f'python evaluate.py --config config/config.yaml --checkpoint {BEST_CKPT}')


In [ ]:
# ============================================================
# CELL 8: Nen va export checkpoint (tai ve tu Kaggle Output tab)
# ============================================================
import os, shutil
from datetime import datetime

OUTPUT_DIR = '/kaggle/working'
CKPT_DIR   = f'{REPO_DIR}/checkpoints'
ts         = datetime.now().strftime('%Y%m%d_%H%M')
zip_base   = f'{OUTPUT_DIR}/mvisar_ckpt_{ts}'

if os.path.exists(CKPT_DIR):
    shutil.make_archive(zip_base, 'zip', CKPT_DIR)
    zip_file = zip_base + '.zip'
    size_mb  = os.path.getsize(zip_file) / 1e6
    print(f'Checkpoints nen xong: {zip_file}')
    print(f'Kich thuoc: {size_mb:.1f} MB')
    print('Tai ve tu Kaggle > Output tab')
else:
    print(f'Khong tim thay checkpoints: {CKPT_DIR}')

print('\n=== Files trong Kaggle Output ===')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f'  {fname}: {size_mb:.1f} MB')
